# 💰 Smart Expense Categorization & Spending Prediction
### Machine Learning Techniques — Mini Project
---
**Objective:** Use ML to automatically categorize expenses and predict spending amounts.

**Models used:**
- 🌲 Random Forest **Classifier** → predicts expense *category*
- 📈 Random Forest **Regressor**  → predicts expense *amount*

## 📦 Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble        import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing   import LabelEncoder
from sklearn.metrics         import (accuracy_score, classification_report,
                                      mean_absolute_error, r2_score)
import matplotlib.ticker as mticker

print('✅ Libraries imported successfully')

## 📂 Step 2 — Load Dataset

In [ ]:
df = pd.read_csv('../data/expenses.csv', parse_dates=['date'])
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
# Basic statistics
print('Category distribution:')
print(df['category'].value_counts())
print('\nAmount statistics:')
print(df['amount'].describe())

## 🧹 Step 3 — Clean Data

In [ ]:
# Check for nulls
print('Null values:')
print(df.isnull().sum())

# Clean
df = df.drop_duplicates()
df = df.dropna(subset=['amount', 'category', 'description'])
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
df = df[df['amount'] > 0].reset_index(drop=True)
print(f'\n✅ Clean dataset size: {len(df)} rows')

## 🔧 Step 4 — Feature Engineering

In [ ]:
# Extract date features
df['day_of_week']  = df['date'].dt.dayofweek
df['month']        = df['date'].dt.month
df['day_of_month'] = df['date'].dt.day
df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)

# Text feature
df['desc_len'] = df['description'].str.split().str.len()

# Encode payment method
le_payment = LabelEncoder()
df['payment_enc'] = le_payment.fit_transform(df['payment_method'])

# Encode category (target)
le_cat = LabelEncoder()
df['category_enc'] = le_cat.fit_transform(df['category'])

print('Category encoding:')
for i, c in enumerate(le_cat.classes_):
    print(f'  {i} → {c}')

df[['description','amount','category','day_of_week','month',
    'is_weekend','desc_len','payment_enc','category_enc']].head()

## 🌲 Step 5 — Classification Model (Predict Category)

In [ ]:
feature_cols = ['day_of_week','month','day_of_month','is_weekend','desc_len','payment_enc']
X     = df[feature_cols]
y_cat = df['category_enc']
y_amt = df['amount']

X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat, test_size=0.25, random_state=42, stratify=y_cat
)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.2%}')
print(classification_report(y_test, y_pred, target_names=le_cat.classes_))

## 📈 Step 6 — Regression Model (Predict Amount)

In [ ]:
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X, y_amt, test_size=0.25, random_state=42
)

reg = RandomForestRegressor(n_estimators=100, random_state=42)
reg.fit(X_tr2, y_tr2)

y_pred2 = reg.predict(X_te2)
mae  = mean_absolute_error(y_te2, y_pred2)
r2   = r2_score(y_te2, y_pred2)
print(f'MAE     : ₹{mae:.2f}')
print(f'R² Score: {r2:.4f}')

# Comparison table
compare = pd.DataFrame({'Actual': y_te2.values, 'Predicted': y_pred2.round(2)})
compare.head(10)

## 📊 Step 7 — Visualizations

In [ ]:
# ── PIE CHART ─────────────────────────────────────────────────
COLORS = ['#4361ee','#f72585','#4cc9f0','#7209b7','#3a0ca3']
cat_totals = df.groupby('category')['amount'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 6))
ax.pie(cat_totals, labels=cat_totals.index, autopct='%1.1f%%',
       colors=COLORS, startangle=140,
       wedgeprops=dict(edgecolor='white', linewidth=1.5))
ax.set_title('💰 Expense Distribution by Category', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── LINE CHART ────────────────────────────────────────────────
daily = df.groupby('date')['amount'].sum().reset_index().sort_values('date')

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(daily['date'], daily['amount'], color='#4361ee',
        linewidth=2, marker='o', markersize=5, markerfacecolor='#f72585')
ax.fill_between(daily['date'], daily['amount'], alpha=0.12, color='#4361ee')
ax.set_title('📈 Daily Spending Trend', fontsize=13, fontweight='bold')
ax.set_ylabel('Amount (₹)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'₹{x:,.0f}'))
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ── FEATURE IMPORTANCE ────────────────────────────────────────
importances = clf.feature_importances_
indices     = importances.argsort()

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh([feature_cols[i] for i in indices], [importances[i] for i in indices],
        color='#4361ee')
ax.set_title('🔍 Feature Importance (Classifier)', fontsize=12, fontweight='bold')
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 🔮 Step 8 — Sample Predictions

In [ ]:
# New expense: weekly grocery (Tuesday, March)
new_expense = pd.DataFrame([{
    'day_of_week': 1, 'month': 3, 'day_of_month': 5,
    'is_weekend': 0, 'desc_len': 6, 'payment_enc': 3
}])

cat_enc   = clf.predict(new_expense)[0]
category  = le_cat.inverse_transform([cat_enc])[0]
proba     = clf.predict_proba(new_expense)[0].max()
predicted = reg.predict(new_expense)[0]

print(f'Description : Weekly grocery shopping')
print(f'Category    : {category}  (confidence: {proba:.1%})')
print(f'Amount est. : ₹{predicted:.2f}')